In [1]:
!apt-get install ncbi-blast+ -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  ncbi-data
The following NEW packages will be installed:
  ncbi-blast+ ncbi-data
0 upgraded, 2 newly installed, 0 to remove and 35 not upgraded.
Need to get 15.8 MB of archives.
After this operation, 71.8 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 ncbi-data all 6.1.20170106+dfsg1-9 [3,519 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 ncbi-blast+ amd64 2.12.0+ds-3build1 [12.3 MB]
Fetched 15.8 MB in 1s (13.4 MB/s)
Selecting previously unselected package ncbi-data.
(Reading database ... 126374 files and directories currently installed.)
Preparing to unpack .../ncbi-data_6.1.20170106+dfsg1-9_all.deb ...
Unpacking ncbi-data (6.1.20170106+dfsg1-9) ...
Selecting previously unselected package ncbi-blast+.
Preparing to unpack .../ncbi-blast+_2.12.0+ds-3build1_amd64.deb .

In [2]:
# 1. Install Biopython
!pip install biopython

# 2. Download Swiss-Prot FASTA
!wget ftp://ftp.ncbi.nlm.nih.gov/blast/db/FASTA/swissprot.gz

# 3. Unzip
!gunzip swissprot.gz


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 52.8 MB/s eta 0:00:00
--2025-09-15 13:16:27--  ftp://ftp.ncbi.nlm.nih.gov/blast/db/FASTA/swissprot.gz
           => ‘swissprot.gz’
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.7, 130.14.250.11, 130.14.250.12, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.7|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /blast/db/FASTA ... done.
==> SIZE swissprot.gz ... 143234298
==> PASV ... done.    ==> RETR swissprot.gz ... done.
Length: 143234298 (137M) (unauthoritative)

swissprot.gz        100%[===================>] 136.60M  20.1MB/s    in 6.8s    

2025-09-15 13:16:36 (20.2 MB/s) - ‘swissprot.gz’ saved [143234298]



In [1]:
from Bio import SeqIO
import re
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter , defaultdict
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch
from tensorflow.keras.preprocessing.sequence import pad_sequences
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

In [2]:
# def extract_protein_info(header):
#     """Extract detailed information from FASTA header"""
#     info = {
#         'accession': None,
#         'protein_name': None,
#         'organism': None,
#         'gene_name': None,
#         'organism_code': None
#     }

#     # Extract accession (first part before |)
#     acc_match = re.search(r'^(\w+\|\w+\|\w+)', header)
#     if acc_match:
#         info['accession'] = acc_match.group(1)

#     # Extract protein name (text before OS=)
#     protein_match = re.search(r'^[^|]*\|[^|]*\|([^=]+?)(?:\s+OS=|$)', header)
#     if protein_match:
#         info['protein_name'] = protein_match.group(1).strip()

#     # Extract organism with OS=
#     os_match = re.search(r'OS=([^=]+?)(?:\s+OX=|$)', header)
#     if os_match:
#         info['organism'] = os_match.group(1).strip()

#     # Extract gene name with GN=
#     gn_match = re.search(r'GN=([^=]+?)(?:\s+[A-Z]{2}=|$)', header)
#     if gn_match:
#         info['gene_name'] = gn_match.group(1).strip()

#     # Also try bracket format for organism
#     if not info['organism']:
#         bracket_match = re.search(r'\[([^\]]+)\]', header)
#         if bracket_match:
#             info['organism'] = bracket_match.group(1)

#     return info

# def classify_organism_type(species_name):
#     """Basic organism type classification based on common patterns"""
#     if not species_name:
#         return 'Unknown'

#     species_lower = species_name.lower()

#     # Common bacterial indicators
#     bacteria_indicators = ['escherichia', 'bacillus', 'streptococcus', 'staphylococcus',
#                           'pseudomonas', 'salmonella', 'clostridium', 'klebsiella']

#     # Common eukaryotic indicators
#     eukaryote_indicators = ['homo', 'mus', 'rattus', 'drosophila', 'caenorhabditis',
#                            'arabidopsis', 'saccharomyces', 'schizosaccharomyces']

#     # Common archaeal indicators
#     archaea_indicators = ['methanococcus', 'thermococcus', 'pyrococcus', 'sulfolobus']

#     for indicator in bacteria_indicators:
#         if indicator in species_lower:
#             return 'Bacteria'

#     for indicator in eukaryote_indicators:
#         if indicator in species_lower:
#             return 'Eukaryota'

#     for indicator in archaea_indicators:
#         if indicator in species_lower:
#             return 'Archaea'

#     return 'Unknown'

# def fasta_to_csv(fasta_file, output_csv, min_samples=10, max_sequence_length=2000):
#     """
#     Convert FASTA file to CSV with comprehensive protein information

#     Args:
#         fasta_file: Path to FASTA file
#         output_csv: Path for output CSV
#         min_samples: Minimum sequences per species to include
#         max_sequence_length: Maximum sequence length to include
#     """

#     print("Processing FASTA file...")

#     data = []
#     sequence_counter = 0

#     # First pass: collect all data
#     for record in SeqIO.parse(fasta_file, "fasta"):
#         sequence_counter += 1
#         if sequence_counter % 10000 == 0:
#             print(f"Processed {sequence_counter} sequences...")

#         seq_str = str(record.seq)
#         header = record.description

#         # Skip very long sequences to avoid memory issues
#         if len(seq_str) > max_sequence_length:
#             continue

#         info = extract_protein_info(header)

#         species = info['organism']
#         if not species:
#             species_match = re.search(r'\\[([^\\]]+)\\]', header)
#             if species_match:
#                 species = species_match.group(1)
#             else:
#                 species = 'Unknown'


#         species = species.strip()

#         data.append({
#             'sequence_id': record.id,
#             'sequence': seq_str,
#             'species': species,
#             'sequence_length': len(seq_str),
#             'header': header,
#             'accession': info['accession'],
#             'protein_name': info['protein_name'],
#             'gene_name': info['gene_name'],
#             'organism_type': classify_organism_type(species)
#         })

#     print(f"\\nCollected {len(data)} sequences")

#     df = pd.DataFrame(data)


#     if min_samples > 1:
#         print(f"Filtering species with at least {min_samples} samples...")
#         species_counts = df['species'].value_counts()
#         valid_species = species_counts[species_counts >= min_samples].index
#         df = df[df['species'].isin(valid_species)].copy()

#         print(f"After filtering: {len(df)} sequences across {len(valid_species)} species")

#     df['gc_content'] = df['sequence'].apply(lambda x: (x.count('G') + x.count('C')) / len(x) if len(x) > 0 else 0)


#     amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
#     for aa in amino_acids:
#         df[f'{aa}_count'] = df['sequence'].apply(lambda x: x.count(aa))
#         df[f'{aa}_freq'] = df[f'{aa}_count'] / df['sequence_length']


#     df['unique_aa_count'] = df['sequence'].apply(lambda x: len(set(x)))


#     print(f"Saving to {output_csv}...")
#     df.to_csv(output_csv, index=False)


#     print("\\n=== DATASET SUMMARY ===")
#     print(f"Total sequences: {len(df)}")
#     print(f"Unique species: {df['species'].nunique()}")
#     print(f"Sequence length range: {df['sequence_length'].min()} - {df['sequence_length'].max()}")
#     print(f"Average sequence length: {df['sequence_length'].mean():.1f}")

#     print("\\nTop 10 species by count:")
#     print(df['species'].value_counts().head(10))

#     print("\\nOrganism type distribution:")
#     print(df['organism_type'].value_counts())

#     return df


# if __name__ == "__main__":

#     df = fasta_to_csv(
#         fasta_file="swissprot",
#         output_csv="protein_sequences.csv",
#         min_samples=10,
#         max_sequence_length=2000
#     )

#     print("\\nConversion complete!")
#     print(f"CSV saved with {len(df)} sequences")
#     print(f"Columns: {list(df.columns)}")

In [3]:
df=pd.read_csv('new_protein_sequences.csv')

In [4]:
aa_vocab = {
    'A': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5,
    'G': 6, 'H': 7, 'I': 8, 'K': 9, 'L': 10,
    'M': 11, 'N': 12, 'P': 13, 'Q': 14, 'R': 15,
    'S': 16, 'T': 17, 'V': 18, 'W': 19, 'Y': 20
}


In [5]:
UNK_TOKEN = len(aa_vocab)

def encode_sequence(seq, vocab=aa_vocab):
    """Convert protein sequence string to integer-encoded list"""
    return [vocab.get(aa, UNK_TOKEN) for aa in seq]

# Encode all sequences in df['sequence']
df['encoded_sequence'] = df['sequence'].apply(encode_sequence)

# Pad sequences to a fixed length (e.g. 2000 like you used earlier)
MAX_LEN = int(np.percentile(df['sequence_length'], 95))
X = pad_sequences(df['encoded_sequence'], maxlen=MAX_LEN, padding='post', truncating='post')

print("Shape of X (encoded + padded):", X.shape)

Shape of X (encoded + padded): (452259, 883)


In [6]:
sequence_features = df['encoded_sequence']   # integer encoded sequences
numeric_features = df[['sequence_length', 'gc_content', 'unique_aa_count']]

target = df['organism_type']
MAX_LEN = 2000  # fixed sequence length
X_num = numeric_features.values


label_encoder = LabelEncoder()
y = label_encoder.fit_transform(target)

X_seq_trainval, X_seq_test, X_num_trainval, X_num_test, y_trainval, y_test = train_test_split(
    X, X_num, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: from train+val → 80% train, 20% val
X_seq_train, X_seq_val, X_num_train, X_num_val, y_train, y_val = train_test_split(
    X_seq_trainval, X_num_trainval, y_trainval, test_size=0.2, random_state=42, stratify=y_trainval
)

print("Train sequence shape:", X_seq_train.shape)
print("Test sequence shape:", X_seq_test.shape)
print("Train numeric shape:", X_num_train.shape)
print("Test numeric shape:", X_num_test.shape)
print("Train labels shape:", y_train.shape)
print("Test labels shape:", y_test.shape)
print("Classes:", label_encoder.classes_)

Train sequence shape: (289445, 883)
Test sequence shape: (90452, 883)
Train numeric shape: (289445, 3)
Test numeric shape: (90452, 3)
Train labels shape: (289445,)
Test labels shape: (90452,)
Classes: ['Archaea' 'Bacteria' 'Eukaryota' 'Unknown']


In [7]:
class ProteinDataset(Dataset):
    def __init__(self, sequences, numeric_features, labels):
        self.sequences = sequences  # list of integer lists
        self.numeric_features = numeric_features.astype('float32')
        self.labels = labels.astype('int64')

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        seq = torch.tensor(self.sequences[idx], dtype=torch.long)
        num = torch.tensor(self.numeric_features[idx], dtype=torch.float)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return seq, num, label

In [8]:
class ProteinCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, num_numeric_features):
        super().__init__()
        # Sequence branch
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv1 = nn.Conv1d(embed_dim, 128, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(128, 256, kernel_size=5, padding=2)
        self.pool = nn.AdaptiveAvgPool1d(1)

        # Numeric branch
        self.numeric_fc = nn.Sequential(
            nn.Linear(num_numeric_features, 64),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Merge branches
        self.fc = nn.Sequential(
            nn.Linear(256 + 64, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, sequence, numeric_features):
        # CNN branch
        x = self.embedding(sequence)         # [batch, seq_len, embed_dim]
        x = x.transpose(1, 2)                # [batch, embed_dim, seq_len]
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)         # [batch, 256]

        # Numeric branch
        n = self.numeric_fc(numeric_features)  # [batch, 64]

        # Concatenate
        out = torch.cat([x, n], dim=1)       # [batch, 320]
        out = self.fc(out)
        return out

In [9]:
model = ProteinCNN(vocab_size=len(aa_vocab) + 1, embed_dim=128, num_classes=len(label_encoder.classes_), num_numeric_features=numeric_features.shape[1])

In [10]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(), lr=0.001)

In [11]:
batch_size = 32

train_dataset = ProteinDataset(X_seq_train, X_num_train, y_train)
val_dataset = ProteinDataset(X_seq_val, X_num_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)


In [13]:
batch_size=128

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

def train_epoch(model, train_loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for batch_x_seq, batch_x_num, batch_y in train_loader:
        batch_x_seq, batch_x_num, batch_y = batch_x_seq.to(device), batch_x_num.to(device), batch_y.to(device)  # move both to GPU

        optimizer.zero_grad()
        outputs = model(batch_x_seq, batch_x_num)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def validate(model, val_loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x_seq, batch_x_num, batch_y in val_loader:
            batch_x_seq, batch_x_num, batch_y = batch_x_seq.to(device), batch_x_num.to(device), batch_y.to(device)  # move both to GPU
            outputs = model(batch_x_seq, batch_x_num)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()

    accuracy = 100 * correct / total
    return total_loss / len(val_loader), accuracy

# Training loop
epochs = 50
for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = validate(model, val_loader, criterion)

    print(f'Epoch {epoch}: Train Loss: {train_loss:.4f}, '
          f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')